In [54]:
# Basic Libraries / Import statements

import numpy as np

import plotly.graph_objects as go 

In [55]:
# Target: assumed to be cruising in straight line at high altitude
target_state = np.array([0.0, 0.0 , 10000.0, 300.0, 0.0, 0.0])

# For basic assume air to ground at a upward angle of attack 
interceptor_state = np.array([2000.0, 500.0, 0.0, -150.0, -50.0, 600.0])

# Basic physics 
dt = 0.1  

# Sim for 30 seconds just to verify 
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))

# Simple Euler integration (constant velocity for this initial test)

for i in range(steps):
    # Recording positions
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    # Update positions based on velocity
    target_state[0:3] += target_state[3:6] * dt
    interceptor_state[0:3] += interceptor_state[3:6] * dt

# Visualize
fig = go.Figure()

# Plot Target Trajectory
fig.add_trace(go.Scatter3d(
    x=target_path[:,0], y=target_path[:,1], z=target_path[:,2],
    mode='lines+markers',
    marker=dict(size=2, color='red'),
    line=dict(color='red', width=2),
    name='Target'
))

# Plot Interceptor Trajectory
fig.add_trace(go.Scatter3d(
    x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2],
    mode='lines+markers',
    marker=dict(size=2, color='blue'),
    line=dict(color='blue', width=2),
    name='Interceptor'
))

fig.update_layout(
    title='Phase 1: 3D Kinematic Environment',
    scene=dict(
        xaxis_title='X Position (m)',
        yaxis_title='Y Position (m)',
        zaxis_title='Altitude Z (m)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


In [56]:

# Intial states, forcing collision 
target_state = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 

# Interceptor velocity 
# adjusted to ensure an interception at t=20s

interceptor_state = np.array([2000.0, 500.0, 0.0, 200.0, -25.0, 500.0])

dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))

# Sim and Distance Tracking 
distances = np.zeros(steps)

for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    # Calculate distance between the two missiles at this exact time step
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    target_state[0:3] += target_state[3:6] * dt
    interceptor_state[0:3] += interceptor_state[3:6] * dt

# Interception point
hit_radius = 50.0 # meters
min_dist_idx = np.argmin(distances)
closest_distance = distances[min_dist_idx]
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

print(f"--- Simulation Results ---")
print(f"Closest Approach: {closest_distance:.2f} meters")
if closest_distance <= hit_radius:
    print(f"STATUS: SUCCESSFUL INTERCEPTION at t = {time_of_interception:.1f}s")
    print(f"Coordinates (x, y, z): {interception_coords}")
else:
    print(f"STATUS: MISS")


fig = go.Figure()

# Plot static tracks (faded lines so you can see the path before the dot gets there)
fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.2)', width=2), name='Target Track'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.2)', width=2), name='Interceptor Track'))

# Plot the specific interception point as a large marker
if closest_distance <= hit_radius:
    fig.add_trace(go.Scatter3d(
        x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]],
        mode='markers', marker=dict(size=8, color='yellow', symbol='diamond'), name='Point of Impact'
    ))

# Create the animated dots (starting at t=0)
fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

# Build frames for the animation (skipping frames so the browser doesn't freeze)
frame_skip = 5 
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[3, 4], # Update only the animated dots (traces 3 and 4)
        name=f'frame_{k}'
    ))
fig.frames = frames

# Add Play/Pause buttons
fig.update_layout(
    title='Phase 1: Animated Kinematics & Interception',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)'),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

--- Simulation Results ---
Closest Approach: 0.00 meters
STATUS: SUCCESSFUL INTERCEPTION at t = 20.0s
Coordinates (x, y, z): [ 6000.     0. 10000.]


In [57]:
target_state = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 

# Re-tuned: interceptor needs more vz to fight gravity, 
# and aims at the physics-adjusted intercept point
interceptor_state = np.array([2000.0, 500.0, 0.0, 190.0, -25.0, 490.0])

dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    t = i * dt
    target_state = rk4_step(target_state, t, dt, missile_dynamics)
    interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics)

# Diagnostic: print where each missile actually is at t=20s
t20 = int(20.0 / dt)
print(f"Target at t=20s:      {target_path[t20]}")
print(f"Interceptor at t=20s: {interceptor_path[t20]}")
print(f"Gap at t=20s: {np.linalg.norm(target_path[t20] - interceptor_path[t20]):.2f}m")

Target at t=20s:      [5501.57130773    0.         8148.18409232]
Interceptor at t=20s: [5351.94975412   58.95397972 6823.16607588]
Gap at t=20s: 1334.74m


# **Phase one working so far**
### Still have to do these updates pre phase 2
#### *Updates:*

- Physics behind actual trajectory and position -> need to understand / do 
- Vector Positioning in relation to missile dynamics 
- Collision Geometry 
- Point of Impact physics, i.e. one collision what happens 
- Open sourced missile information 
- Actual interception math  -> test against forced values 
- Actual missile atypcial flight patterns and normal/varied impact patterns 
- Basic enviorment mapping 
- Atypical Launch :
    Patterns 
    Timings for intercept to target 
    Cruise patterns 

## Updates Planned 

- using a self made Rk4 
- Rk4 -> is a Runge Kutta (4th order)
- Numerical Method used for solving differential equations 

- The physics gives a rate of change
- but the sim actually need positional information 
- Since it is very difficult to solve for it makes more senst to integrate numericaly 


In [58]:

def rk4_step(state, t, dt, derivatives_fn):
    """
    Standard 4th-Order Runge-Kutta integrator.
    Updates the state vector for a single time step.
    """
    k1 = derivatives_fn(state, t)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt)
    k4 = derivatives_fn(state + dt * k3, t + dt)
    
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def missile_dynamics(state, t):
    vel = state[3:6]
    
    gravity_accel = np.array([0.0, 0.0, -9.81])
    
    # Fixed: drag_coeff was 0.005 — roughly 100x too strong
    # This was killing horizontal velocity and preventing interceptor from climbing
    drag_coeff = 0.00003  
    drag_accel = -drag_coeff * vel * np.linalg.norm(vel)
    
    accel = gravity_accel + drag_accel
    return np.concatenate((vel, accel))  

In [59]:
# Bumped the interceptor's initial Z-velocity (vz) up to 600.0
target_state      = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 
interceptor_state = np.array([2000.0, 500.0, 0.0, 200.0, -25.0, 500.0])


dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

# --- 3. Simulation Loop (RK4 Integration) ---
for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    t = i * dt
    
    target_state = rk4_step(target_state, t, dt, missile_dynamics)
    interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics)

# --- 4. Interception Analysis ---
hit_radius = 50.0 
min_dist_idx = np.argmin(distances)
closest_distance = distances[min_dist_idx]
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

print(f"--- Simulation Results ---")
print(f"Closest Approach: {closest_distance:.2f} meters")
if closest_distance <= hit_radius:
    print(f"STATUS: SUCCESSFUL INTERCEPTION at t = {time_of_interception:.1f}s")
    print(f"Coordinates (x, y, z): {interception_coords}")
else:
    print(f"STATUS: MISS")

# --- 5. 3D Animated Visualization ---
# --- 5. 3D Animated Visualization ---
fig = go.Figure()

# 0: Target Track
fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.2)', width=2), name='Target Track'))
# 1: Interceptor Track
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.2)', width=2), name='Interceptor Track'))

# 2: Point of Impact (Conditional)
if closest_distance <= hit_radius:
    fig.add_trace(go.Scatter3d(
        x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]],
        mode='markers', marker=dict(size=8, color='yellow', symbol='diamond'), name='Point of Impact'
    ))

# Dynamically find the indices for the animated dots
base_traces = len(fig.data)
target_idx = base_traces
interceptor_idx = base_traces + 1

fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

frame_skip = 5 
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[target_idx, interceptor_idx], # Dynamically targeting the correct dots
        name=f'frame_{k}'
    ))
fig.frames = frames

fig.update_layout(
    title='Phase 1: Animated Kinematics (RK4 Dynamics)',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)'),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

--- Simulation Results ---
Closest Approach: 252.11 meters
STATUS: MISS


In [60]:
#Updates Jun 1

In [61]:
import numpy as np
import plotly.graph_objects as go

# --- 1. Math & Geometry ---
def compute_cpa(pos1, vel1, pos2, vel2):
    dp = pos1 - pos2
    dv = vel1 - vel2
    # Add epsilon to prevent division by zero if velocities are identical
    dv_sq = max(np.dot(dv, dv), 1e-8) 
    t_cpa = -np.dot(dp, dv) / dv_sq
    
    # If t_cpa is negative, the closest point is in the past
    if t_cpa < 0:
        return 0.0, np.linalg.norm(dp)
        
    pos1_cpa = pos1 + vel1 * t_cpa
    pos2_cpa = pos2 + vel2 * t_cpa
    miss_dist = np.linalg.norm(pos1_cpa - pos2_cpa)
    return t_cpa, miss_dist

def proportional_navigation(interceptor_state, target_state, N=4.0):
    r_vec = target_state[0:3] - interceptor_state[0:3]
    v_rel = target_state[3:6] - interceptor_state[3:6]
    
    r = max(np.linalg.norm(r_vec), 1e-6)
    r_hat = r_vec / r
    
    omega = np.cross(r_vec, v_rel) / (r ** 2)
    
    v_mag = np.linalg.norm(interceptor_state[3:6])
    accel_cmd = N * v_mag * np.cross(omega, r_hat)
    
    # Cap the maximum G-force the interceptor can pull (e.g., 30 Gs)
    max_accel = 30.0 * 9.81
    if np.linalg.norm(accel_cmd) > max_accel:
        accel_cmd = (accel_cmd / np.linalg.norm(accel_cmd)) * max_accel
        
    return accel_cmd

# --- 2. Physics Engine ---
def missile_dynamics(state, t, accel_cmd=np.zeros(3)):
    vel = state[3:6]
    speed = np.linalg.norm(vel)
    
    gravity_accel = np.array([0.0, 0.0, -9.81])
    drag_accel = -0.00003 * vel * speed
    
    # accel_cmd is injected from the guidance law or target evasive logic
    accel = gravity_accel + drag_accel + accel_cmd
    return np.concatenate((vel, accel))

def rk4_step(state, t, dt, derivatives_fn, accel_cmd=np.zeros(3)):
    k1 = derivatives_fn(state, t, accel_cmd)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt, accel_cmd)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt, accel_cmd)
    k4 = derivatives_fn(state + dt * k3, t + dt, accel_cmd)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

In [62]:
# --- 1. Initial States ---
target_state = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 
interceptor_state = np.array([2000.0, 500.0, 0.0, 200.0, -25.0, 530.0])

dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

# --- 2. Simulation Loop with Active Guidance ---
for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    t = i * dt
    
    # Target Evasive Maneuver: 3g lateral wave starting at 5 seconds
    target_accel = np.zeros(3)
    if t > 5.0:
        target_accel[1] = 30.0 * np.sin(0.5 * t) 
        
    # Interceptor Guidance: Calculate PN command
    interceptor_accel = proportional_navigation(interceptor_state, target_state, N=4.0)
    
    # Step Physics forward
    target_state = rk4_step(target_state, t, dt, missile_dynamics, accel_cmd=target_accel)
    interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics, accel_cmd=interceptor_accel)

# --- 3. CPA Hit Analysis ---
hit_radius = 20.0 
min_dist_idx = np.argmin(distances)
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

# Run geometric CPA on the closest recorded states for sub-timestep precision
t_cpa, precise_miss_dist = compute_cpa(
    target_path[min_dist_idx], target_state[3:6], 
    interceptor_path[min_dist_idx], interceptor_state[3:6]
)

print(f"--- ProNav Simulation Results ---")
print(f"Closest Recorded Gap: {distances[min_dist_idx]:.2f} meters")
print(f"Geometric CPA Miss:   {precise_miss_dist:.2f} meters")

if precise_miss_dist <= hit_radius:
    print(f"STATUS: TARGET DESTROYED at t = {time_of_interception:.1f}s")
else:
    print(f"STATUS: MISS")

# --- 4. Plotting ---
fig = go.Figure()

fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.4)', width=3), name='Target (Evasive)'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.4)', width=3), name='Interceptor (ProNav)'))

if precise_miss_dist <= hit_radius:
    fig.add_trace(go.Scatter3d(
        x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]],
        mode='markers', marker=dict(size=10, color='yellow', symbol='x'), name='Detonation Point'
    ))

# Find base traces for animation
base_traces = len(fig.data)
target_idx = base_traces
interceptor_idx = base_traces + 1

fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target Marker'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor Marker'))

frame_skip = 2 # Lowered skip for smoother visual tracking of maneuvers
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[target_idx, interceptor_idx],
        name=f'frame_{k}'
    ))
fig.frames = frames

fig.update_layout(
    title='Phase 1: Proportional Navigation vs Maneuvering Target',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)'),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=20, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

--- ProNav Simulation Results ---
Closest Recorded Gap: 20.43 meters
Geometric CPA Miss:   2.58 meters
STATUS: TARGET DESTROYED at t = 21.7s


# **Phase sim updates**

#### *Updates:*

- Closest Recorded Gap -> 20.43 meters, discrete mesurement
- That is the smallest distance captured at one of the 0.1s time stamps
- The Geometric CPA miss 2.58 meters was the mathematically computed true closest point of approach 
- The CPA function extrapolating exactly when the two trajetories converge based on relative velocity 
- That is considered the real miss, the real world estimate of within the kill radius would be around 10-50m (fragmentation as well)
- The actual gap of 20.43 vs 2.58 -> means the intercept happened very fast, the large gap means there is a high closing velocty 
- Physically correct for head on or nbut ear hed on intercept 


# Phase 4: Machine Learning Integration Architecture

This is highly feasible and represents the exact architecture used in modern Integrated Air and Missile Defense (IAMD) systems. This effectively designs a simplified version of a system like AEGIS or the Patriot Information Coordination Central (ICC).

Implementing this after Phase 2 and 3 is the perfect sequence because those earlier phases are strictly required to generate the synthetic data the machine learning models will need to train on.

This "Phase 4" machine learning update breaks down into two distinct computational problems.

## 1. The Prediction Engine: "Where is it going?"

Before the system can decide *how* to shoot the target down, it has to accurately predict the target's trajectory and intended impact zone based on limited radar snapshots (e.g., $x, y, z, v_x, v_y, v_z$ over a few seconds).

* **The ML Approach:** Time-series forecasting. State vectors are fed into a Recurrent Neural Network (RNN) or an LSTM (Long Short-Term Memory) network.
* **Clarifying "Reverse Bias":** In circuit design, a reverse bias acts as a gate to block current (like in a diode). When working backward from a goal, the ML equivalents are **Backpropagation** (how the network learns by calculating the error gradient backward from the output) or **Inverse Kinematics** (working backward from an impact point to determine the launch origin).
* **The Alternative (Classic):** Militaries traditionally use Extended Kalman Filters (EKF) for this rather than deep learning, as EKFs are mathematically provable and computationally cheap. However, training an LSTM to predict the terminal phase of a maneuvering target is a high-level research application.

## 2. Dynamic Weapon Target Assignment (DWTA): "Which silo fires?"

Once the LSTM predicts the target's trajectory and impact point across the geospatial map (from Phase 3), the system must choose which Surface-to-Air Missile (SAM) silo to activate.

This is an optimization problem to maximize interception probability while minimizing debris cost.

* **The ML Approach:** **Reinforcement Learning (RL)**. This utilizes a Deep Q-Network (DQN) or Proximal Policy Optimization (PPO) agent.
* **How it Works:** The RL agent looks at the "State" (Target trajectory, Silo 1 status, Silo 2 status, Silo 3 status). It takes an "Action" (Fire Silo 2). The simulation runs the engagement. If Silo 2 hits the target and the debris falls in the ocean, the agent gets a massive positive Reward. If Silo 2 misses, or the debris falls on a city, the agent gets a massive negative penalty (cost).
* **Training:** The simulation is run hundreds of thousands of times with random target launches, allowing the RL agent to learn which silo geometry provides the best kinematic advantage.

## Why Phases 2 and 3 are Strict Prerequisites

An ML algorithm cannot be trained without a massive dataset.

* **Phase 2 (Actual Physics)** acts as the synthetic data generator. The RK4 simulation loop will run thousands of times, randomizing the target's speed, launch angle, and maneuvers, to generate the CSV files the LSTM will train on.
* **Phase 3 (Geospatial & Cost Mapping)** acts as the RL agent's "Reward Function." The agent cannot learn to minimize cost until the cost function exists in the environment.
